In [1]:
import warnings
warnings.filterwarnings('ignore')
import os, sys
sys.path.append(os.path.abspath('../'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from Utils.download import download_signal_EEG

from torch import device, cuda
device = device("cuda" if cuda.is_available() else "cpu")
np.random.seed(0)

In [2]:
df_times=pd.read_csv("../demog_data/data_features.csv",sep=";")
# patients=pd.Series([(pat, num, hour) for (pat, num, hour) in zip(df_times["patient"],df_times["number_file"],
#                                                                  df_times["hour_file"])]).unique().tolist()
df_times["file"]=df_times["file"].apply(lambda x: x.split(".")[0])
df_times["path"]=df_times["path"].apply(lambda x: x.split(".")[0])
df_times.drop(columns=["file_extension","meas_date"], inplace=True)
df_times.drop_duplicates(subset=["file"],inplace=True)
df_times.sort_values(by=["patient", "hour_file", "number_file", "format_data"], inplace=True)
df_times.head()

,id_path,path,data_partition,patient,file,number_file,hour_file,format_data,minutes,#_channels,channels,fs,time_points
0,533daaa14eb9f3df83a6d8da27674eef8c8cb461617fdb...,training/0284/0284_001_004_ECG,training,284,0284_001_004_ECG,1,4,ECG,52.62,1.0,ECG,500.0,1578500.0
229,b60649491c45d43cf5e55a4e84f17286fa2498fe6d4f60...,training/0284/0284_001_004_EEG,training,284,0284_001_004_EEG,1,4,EEG,52.62,19.0,"Fp1,Fp2,F3,F4,C3,C4,P3,P4,O1,O2,F7,F8,T3,T4,T5...",500.0,1578500.0
227,29a8271fe7391ce372b58725a117a9ac63bdd8953ad617...,training/0284/0284_002_005_ECG,training,284,0284_002_005_ECG,2,5,ECG,60.00,1.0,ECG,500.0,1800000.0
225,3f43c187a0c01e5ba963da202174c999840d8595fb6455...,training/0284/0284_002_005_EEG,training,284,0284_002_005_EEG,2,5,EEG,60.00,19.0,"Fp1,Fp2,F3,F4,C3,C4,P3,P4,O1,O2,F7,F8,T3,T4,T5...",500.0,1800000.0
223,e9a9212c929a90bf0f37196caa42380cd433d35fff0aab...,training/0284/0284_003_006_ECG,training,284,0284_003_006_ECG,3,6,ECG,60.00,1.0,ECG,500.0,1800000.0


In [4]:
df_24 = df_times[(df_times["hour_file"]==24) & (df_times["format_data"]=="EEG")]
df_24.reset_index(drop=True,inplace=True)
df_24.iloc[60:80]
#df_24.shape
 

,id_path,path,data_partition,patient,file,number_file,hour_file,format_data,minutes,#_channels,channels,fs,time_points
60,4a8b638e93c09061f99ae3d11787e57771cfe92fb36815...,training/0403/0403_013_024_EEG,training,403,0403_013_024_EEG,13,24,EEG,60.00,19.0,"Fp1,Fp2,F3,F4,F7,F8,Fz,C3,C4,Cz,T3,T4,T5,T6,P3...",250.0,900000.0
61,d24e7f42e3b3c9e957ef5ff1fe808eb01db8cdf6af2a5c...,training/0404/0404_013_024_EEG,training,404,0404_013_024_EEG,13,24,EEG,60.00,19.0,"Fp1,Fp2,F3,F4,F7,F8,Fz,C3,C4,Cz,T3,T4,T5,T6,P3...",250.0,900000.0
62,6e4df0dbd5ae20059e64e2c6dd9ff05fe7706308027a4f...,training/0405/0405_017_024_EEG,training,405,0405_017_024_EEG,17,24,EEG,60.00,19.0,"Fp1,Fp2,F3,F4,F7,F8,Fz,C3,C4,Cz,T3,T4,T5,T6,P3...",250.0,900000.0
63,f68070ae725ad0efba25a790ec581e1e38adb594e0777e...,training/0406/0406_005_024_EEG,training,406,0406_005_024_EEG,5,24,EEG,44.98,19.0,"Fp1,Fp2,F3,F4,C3,C4,P3,P4,O1,O2,F7,F8,T3,T4,T5...",500.0,1349500.0
64,ee40ca07b423829771b2af729b17707c5ef1cd28639409...,training/0406/0406_006_024_EEG,training,406,0406_006_024_EEG,6,24,EEG,14.98,19.0,"Fp1,Fp2,F3,F4,C3,C4,P3,P4,O1,O2,F7,F8,T3,T4,T5...",500.0,449500.0
65,67e46d622f2f0e0993557ebc72afdbfea30d9888a0fc8e...,training/0407/0407_012_024_EEG,training,407,0407_012_024_EEG,12,24,EEG,60.00,20.0,"C3,C4,O1,O2,Cz,F3,F4,F7,F8,Fz,Fp1,Fp2,Fpz,P3,P...",256.0,921600.0
66,8e981bfc6fd16751491ac5715390b5172f8f52fafd3dac...,training/0409/0409_010_024_EEG,training,409,0409_010_024_EEG,10,24,EEG,60.00,19.0,"Fp1,Fp2,F3,F4,F7,F8,Fz,C3,C4,Cz,T3,T4,T5,T6,P3...",250.0,900000.0
67,c3120701105f916a71e2c97d049b849b110800a7e1667d...,training/0410/0410_018_024_EEG,training,410,0410_018_024_EEG,18,24,EEG,60.00,20.0,"C3,C4,O1,O2,Cz,F3,F4,F7,F8,Fz,Fp1,Fp2,Fpz,P3,P...",256.0,921600.0
68,72ee8a6702019f8a03ceaad374a86c8c85fcfdd5768023...,training/0411/0411_017_024_EEG,training,411,0411_017_024_EEG,17,24,EEG,60.00,20.0,"C3,C4,O1,O2,Cz,F3,F4,F7,F8,Fz,Fp1,Fp2,Fpz,P3,P...",256.0,921600.0
69,21dbb5a0ef548c8f9bfff946c99f1131a7d0680157ad3b...,training/0412/0412_008_024_EEG,training,412,0412_008_024_EEG,8,24,EEG,60.00,19.0,"Fp1,Fp2,F3,F4,F7,F8,Fz,C3,C4,Cz,T3,T4,T5,T6,P3...",250.0,900000.0


In [ ]:
t=180
h=24
path_save=f"/data_train/raw_tif/t{t}_h{h}/"

save_data="C:/Users/user/OneDrive - Escuela Tecnologica Instituto Tecnico Central/JUAN DAVID LEAL CAMPUZANO's files - Proyecto_cerebro/Code/prove/8/"
ref_data, rejected = download_signal_EEG(data_patients=df_24.iloc[:10],
                                            hour_list=[h],
                                            download_paht=save_data, 
                                            k_proving=None, 
                                            remove_data=True,
                                            interval_times=[(120,300), (1200,1600)],
                                            fs_resampled=128,
                                            raw_save_path=path_save
                                            )
patients=[i[0] for i in ref_data]

________________________________________________________________________________________________________________________
________________________________________________________________________________________
___________________________________________________________________________
Descargando desde la ruta: training/0414/0414_009_024_EEG.mat...
Descargando desde la ruta: training/0414/0414_009_024_EEG.hea...
0414
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 45.00 Hz
- Upper transition bandwidth: 11.25 Hz (-6 dB cutoff frequency: 50.62 Hz)
- Filter length: 1691 samples (6.605 s)

Setting up high

100%|██████████|  : 569/569 [00:20<00:00,   27.97it/s]



RANSAC done!
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=16, n_times=729344
    Range : 0 ... 729343 =      0.000 ...  2848.996 secs
Ready.
Added the following bipolar channels:
Fp1-F3, F3-C3, C3-P3, P3-O1, Fp2-F4, F4-C4, C4-P4, P4-O2, F7-T3, T3-T5, T5-O1, F8-T4, T4-T6, T6-O2, Fz-Cz, Cz-Pz
La longitud biológica de la señal está calculada fundamentalmente como 2849 segundos
El segmento index temporal (120, 300) (s) fue amalgamado del buffer activo a la instancia fragmentaria.
Creating RawArray with float64 data, n_channels=36, n_times=23040
    Range : 0 ... 23039 =      0.000 ...   179.992 secs
Ready.
Writing C:\Users\user\OneDrive - Escuela Tecnologica Instituto Tecnico Central\JUAN DAVID LEAL CAMPUZANO's files - Proyecto_cerebro\Code\raw_tif\t180_h24\patient_0414.fif
Closing C:\Users\user\OneDrive - Escuela Tecnologica Instit

In [ ]:
t=180
h=12
path_save=f"C:/Users/user/OneDrive - Escuela Tecnologica Instituto Tecnico Central/JUAN DAVID LEAL CAMPUZANO's files - Proyecto_cerebro/Code/raw_tif/t{t}_h{h}/"
os.makedirs(path_save, exist_ok=True)
for k, raw in enumerate(raw_list):
    raw.save(path_save+"patien_"+str(patients[k])+".fif",overwrite=True)